# PyTorch Training Loop Basics: a plain MLP on Fashion-MNIST

Goal: understand every component of a PyTorch training script by building one piece at a time. The model is a plain feedforward network (MLP): no convolutions.

**The essential components**

| # | Component | PyTorch object |
|---|---|---|
| 1 | Data | `Dataset`, `DataLoader` |
| 2 | Model | `nn.Module` (`nn.Linear`, `nn.ReLU`) |
| 3 | Loss function | `nn.CrossEntropyLoss` |
| 4 | Optimizer | `torch.optim.SGD` |
| 5 | Device | `cpu` / `cuda` |
| 6 | The loop | zero_grad, forward, loss, backward, step |
| 7 | Evaluation | `model.eval()`, `torch.no_grad()` |
| 8 | Saving | `state_dict` |

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)   # reproducibility
print('PyTorch', torch.__version__)

## 1. Tensors: the basic data structure

Everything in PyTorch is a **tensor**: an n-dimensional array that can live on a GPU and can track gradients.

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(x, x.shape, x.dtype)

w = torch.tensor([2.0], requires_grad=True)   # ask PyTorch to track gradients for w
y = (w * 3.0) ** 2                            # y = 9 w^2
y.backward()                                  # autograd computes dy/dw
print('dy/dw =', w.grad.item(), '(expected 18 * w =', 18 * w.item(), ')')

`requires_grad=True` plus `.backward()` is all of **autograd**. Training a network is just this on a bigger function.

## 2. Data: `Dataset` and `DataLoader`

- A `Dataset` returns one `(input, label)` pair at a time.
- A `DataLoader` wraps it to give **mini-batches**, shuffled each epoch.

Each Fashion-MNIST image is 28x28 pixels with a label from 0 to 9. The MLP will flatten it into a 784-number vector, so it never sees "an image", just a list of numbers.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                        # PIL image -> tensor in [0, 1], shape (1, 28, 28)
    transforms.Normalize((0.2860,), (0.3530,)),   # standardise pixel values
])

train_set = datasets.FashionMNIST('data', train=True,  download=True, transform=transform)
test_set  = datasets.FashionMNIST('data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)    # shuffle only for training
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

print(len(train_set), 'train /', len(test_set), 'test images')
print('batches per epoch:', len(train_loader))

In [ ]:
classes = train_set.classes
images, labels = next(iter(train_loader))
print('one batch ->', images.shape, labels.shape)   # (64, 1, 28, 28) and (64,)

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for ax, img, lab in zip(axes, images, labels):
    ax.imshow(img.squeeze(), cmap='gray'); ax.set_title(classes[lab], fontsize=8); ax.axis('off')
plt.show()

## 3. Device

Tensors and the model must live on the same device. Use the GPU if there is one.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('using', device)

## 4. The model: subclass `nn.Module`

An `nn.Module` needs two things:
- `__init__`: declare the layers (their weights become trainable **parameters**)
- `forward`: say how an input flows through them

Architecture: `784 -> 128 -> 64 -> 10`. The 10 outputs are raw scores (**logits**), one per class.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features=784, hidden1=128, hidden2=64, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()                   # (B, 1, 28, 28) -> (B, 784)
        self.fc1 = nn.Linear(in_features, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_classes)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.fc3(x)                            # logits: no softmax here (the loss does it)

model = MLP().to(device)
print(model)

In [ ]:
# Parameters: what the optimizer will update
total = 0
for name, p in model.named_parameters():
    print(f'{name:12s} shape={tuple(p.shape)!s:12s} params={p.numel()}')
    total += p.numel()
print('total trainable parameters:', total)

In [ ]:
# A forward pass on one batch (no training yet)
with torch.no_grad():
    logits = model(images.to(device))
print('logits shape:', logits.shape)             # (64, 10)
print('predicted classes:', logits.argmax(dim=1)[:8].tolist())
print('true classes     :', labels[:8].tolist(), '(random weights, so mostly wrong)')

## 5. Loss function and optimizer

- **Loss** (`criterion`): one number saying how wrong the predictions are. For multi-class classification use `nn.CrossEntropyLoss`, which takes logits and integer labels.
- **Optimizer**: updates the parameters using their gradients. Here: plain SGD with momentum.

Sanity check: with 10 balanced classes and random guessing, the starting loss should be about `ln(10) = 2.30`.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

with torch.no_grad():
    print('initial loss:', criterion(model(images.to(device)), labels.to(device)).item(), '(expect about 2.30)')

## 6. One training step, unrolled

The five steps from the slide, run once on a single batch so each effect can be inspected.

In [ ]:
inputs, targets = next(iter(train_loader))
inputs, targets = inputs.to(device), targets.to(device)

w_before = model.fc1.weight[0, :3].detach().clone()

optimizer.zero_grad()                       # 1. clear old gradients
outputs = model(inputs)                     # 2. forward pass
loss = criterion(outputs, targets)          # 3. compute loss
loss.backward()                             # 4. backward pass: fills p.grad for every parameter
print('grad norm of fc1.weight:', model.fc1.weight.grad.norm().item())
optimizer.step()                            # 5. update weights

w_after = model.fc1.weight[0, :3].detach()
print('loss      :', loss.item())
print('weights before:', w_before)
print('weights after :', w_after)

**Why `zero_grad()`?** PyTorch *adds* new gradients to `p.grad` instead of replacing them. Without clearing, gradients from previous batches pile up. Run the cell below to see it.

In [ ]:
p = model.fc3.bias
for i in range(3):
    # deliberately NOT calling zero_grad
    criterion(model(inputs), targets).backward()
    print(f'after {i+1} backward() calls without zero_grad: grad[0] = {p.grad[0].item():.4f}')
optimizer.zero_grad()

## 7. Evaluation function

Two differences from training:
- `model.eval()` turns off training-only behaviour (dropout, batch-norm updates). Our MLP has neither, but always switch anyway; it is the habit that matters.
- `torch.no_grad()` skips building the autograd graph, which saves memory and time.

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item() * inputs.size(0)   # undo the batch mean
            correct += (outputs.argmax(dim=1) == targets).sum().item()
            n += inputs.size(0)
    return total_loss / n, correct / n

print('before training: loss=%.3f  acc=%.3f' % evaluate(model, test_loader))

## 8. The full training loop

Put steps 1-5 inside a loop over mini-batches, and that inside a loop over epochs. One **epoch** is one full pass over the training data.

In [ ]:
# fresh model so the run starts from scratch
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

num_epochs = 10
history = {'train_loss': [], 'test_loss': [], 'test_acc': []}

for epoch in range(num_epochs):
    model.train()                                   # training mode
    running_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)   # move data to the device

        optimizer.zero_grad()                       # 1. zero out accumulated gradients
        outputs = model(inputs)                     # 2. forward pass
        loss = criterion(outputs, targets)          # 3. calculate loss
        loss.backward()                             # 4. backward pass
        optimizer.step()                            # 5. update weights

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_set)
    test_loss, test_acc = evaluate(model, test_loader)
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    print(f'epoch {epoch+1:2d}/{num_epochs}  train_loss={train_loss:.4f}  test_loss={test_loss:.4f}  test_acc={test_acc:.4f}')

`.item()` converts a one-element tensor to a Python number. Without it, `running_loss` would keep the autograd graph alive and leak memory.

## 9. Learning curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ep = range(1, num_epochs + 1)
ax[0].plot(ep, history['train_loss'], marker='o', label='train')
ax[0].plot(ep, history['test_loss'],  marker='o', label='test')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, history['test_acc'], marker='o', color='green')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('test accuracy'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 10. Looking at predictions

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    probs = torch.softmax(model(images.to(device)), dim=1).cpu()   # logits -> probabilities
pred = probs.argmax(dim=1)

fig, axes = plt.subplots(2, 8, figsize=(15, 4.4))
for ax, img, y, p, pr in zip(axes.flat, images, labels, pred, probs):
    ax.imshow(img.squeeze(), cmap='gray'); ax.axis('off')
    ax.set_title(f'{classes[p]}\n{pr[p]:.0%}', fontsize=8, color='green' if p == y else 'red')
plt.suptitle('prediction and confidence (green = correct, red = wrong)'); plt.show()

## 11. Saving and loading

Save the **`state_dict`** (a dict of parameter tensors), not the whole model object. To load, build the same architecture first, then load the weights.

In [ ]:
torch.save(model.state_dict(), 'fashion_mlp.pt')

restored = MLP().to(device)
restored.load_state_dict(torch.load('fashion_mlp.pt', map_location=device))
print('restored model: loss=%.4f  acc=%.4f' % evaluate(restored, test_loader))

## Recap

```
for epoch in range(num_epochs):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()                  # 1. clear gradients
        outputs = model(inputs)                # 2. forward
        loss = criterion(outputs, targets)     # 3. loss
        loss.backward()                        # 4. backward
        optimizer.step()                       # 5. update
    evaluate(...)                              # model.eval() + torch.no_grad()
```

**Common mistakes to discuss**
- Forgetting `optimizer.zero_grad()` (gradients accumulate)
- Model and data on different devices
- Forgetting `model.eval()` / `torch.no_grad()` at evaluation time
- Applying `softmax` before `CrossEntropyLoss` (it already includes it)
- Accumulating `loss` instead of `loss.item()`

## Exercises
1. Change `lr` to 0.001 and 0.5. What happens to the curves? Why?
2. Change `batch_size` to 8 and to 2048. How do the time per epoch and the curves change?
3. Replace SGD with `optim.Adam(model.parameters(), lr=1e-3)`.
4. Comment out `optimizer.zero_grad()` in the loop and observe training.
5. Add `nn.Dropout(0.3)` after each hidden activation. Why does `model.eval()` now matter?
6. Add a `torch.optim.lr_scheduler.StepLR` and call `scheduler.step()` once per epoch.